# OULAD - four binary tasks, federated learning, and XAI

This notebook reconstructs the full OULAD pipeline that was missing from the archived experiment: official data download, clickstream and demographic feature engineering, four binary outcomes, centralized training, FedAvg, FedProx, and post-hoc explanations.

Data come from the [Open University Learning Analytics Dataset](https://analyse.kmi.open.ac.uk/open_dataset) ([UCI mirror and CC BY 4.0 citation](https://archive.ics.uci.edu/dataset/349/open+university+learning+analytics+dataset)). The raw archive expands to roughly 464 MB and feature engineering processes 10.6 million VLE records.

Set `AIED_PAPER_MODE=1` for all four tasks with the paper settings. The default is a smoke test. `AIED_OULAD_TASKS=pass-fail` can restrict the tasks; `AIED_RUN_XAI=1` enables explanations.

In [ ]:
import os
import random
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

DEFAULT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REPO_ROOT = Path(os.environ.get("AIED_REPO_ROOT", DEFAULT_ROOT)).resolve()
DATA_DIR = REPO_ROOT / "data" / "oulad"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = REPO_ROOT / "results" / "oulad"
PAPER_MODE = os.environ.get("AIED_PAPER_MODE", "0") == "1"
RUN_XAI = os.environ.get("AIED_RUN_XAI", "0") == "1"
REBUILD_FEATURES = os.environ.get("AIED_REBUILD_FEATURES", "0") == "1"
TASKS = [x.strip() for x in os.environ.get(
    "AIED_OULAD_TASKS", "pass-fail,fail-distinction,pass-distinction,pass-withdrawn"
).split(",") if x.strip()]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CONFIG = {
    "repeats": 10 if PAPER_MODE else 1,
    "central_epochs": 100 if PAPER_MODE else 2,
    "clients": 100 if PAPER_MODE else 3,
    "rounds": 50 if PAPER_MODE else 2,
    "local_epochs": 2 if PAPER_MODE else 1,
    "batch_size": 64,
    "lr": 0.02,
    "mu": 0.01,
    "fast_max_rows": None if PAPER_MODE else 2000,
}
print({"paper_mode": PAPER_MODE, "tasks": TASKS, "device": str(DEVICE), **CONFIG})

## 1. Download and feature engineering

Features follow the implementation cited by the paper:

- clicks by VLE activity type before (`BC`, date < 0) and after (`AC`, date >= 0) presentation start;
- one-hot region, deprivation band, prior education, and age band;
- gender, disability, previous attempts, and studied credits;
- total pre/post-start clicks; and
- clicks one day before, on, and one day after assessment dates.

In [ ]:
OULAD_DOWNLOAD = "https://archive.ics.uci.edu/static/public/349/open+university+learning+analytics+dataset.zip"
REQUIRED_RAW = ["assessments.csv", "studentInfo.csv", "studentVle.csv", "vle.csv"]

def ensure_oulad_raw():
    if all((RAW_DIR / name).exists() for name in REQUIRED_RAW):
        return
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    archive = DATA_DIR / "oulad_349.zip"
    print(f"Downloading {OULAD_DOWNLOAD}")
    urllib.request.urlretrieve(OULAD_DOWNLOAD, archive)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(RAW_DIR)
    missing = [name for name in REQUIRED_RAW if not (RAW_DIR / name).exists()]
    if missing:
        raise FileNotFoundError(f"OULAD archive is missing: {missing}")


def engineer_oulad_features():
    ensure_oulad_raw()
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    info = pd.read_csv(RAW_DIR / "studentInfo.csv")
    vle = pd.read_csv(RAW_DIR / "vle.csv", usecols=["id_site", "activity_type"])
    assessments = pd.read_csv(
        RAW_DIR / "assessments.csv",
        usecols=["code_module", "code_presentation", "date"],
    )
    assessments["date"] = pd.to_numeric(assessments["date"], errors="coerce")
    assessments = assessments.dropna(subset=["date"])
    student_vle = pd.read_csv(
        RAW_DIR / "studentVle.csv",
        usecols=["code_module", "code_presentation", "id_student", "id_site", "date", "sum_click"],
        dtype={"id_student": "int32", "id_site": "int32", "date": "int16", "sum_click": "int32"},
    )
    keys = ["code_module", "code_presentation", "id_student"]
    # The current UCI mirror encodes missing deprivation bands as '?'; the
    # original OULAD release treated them as missing rather than a category.
    info["imd_band"] = info["imd_band"].replace("?", np.nan)

    site_to_activity = vle.drop_duplicates("id_site").set_index("id_site")["activity_type"]
    student_vle["activity_type"] = student_vle["id_site"].map(site_to_activity)
    student_vle["period"] = np.where(student_vle["date"] >= 0, "AC", "BC")
    student_vle["activity_feature"] = student_vle["period"] + "_" + student_vle["activity_type"].astype(str)
    activity = (
        student_vle.groupby(keys + ["activity_feature"], observed=True)["sum_click"]
        .sum().unstack(fill_value=0).reset_index()
    )

    base = info[keys].copy()
    categorical = pd.concat(
        [
            pd.get_dummies(info["region"], dtype="int8"),
            pd.get_dummies(info["imd_band"], dtype="int8"),
            pd.get_dummies(info["highest_education"], dtype="int8"),
            pd.get_dummies(info["age_band"], dtype="int8"),
        ],
        axis=1,
    )
    base = pd.concat([base.reset_index(drop=True), categorical.reset_index(drop=True)], axis=1)
    base["gender"] = (info["gender"] == "F").astype("int8")
    base["disability"] = (info["disability"] == "Y").astype("int8")
    base["num_of_prev_attempts"] = info["num_of_prev_attempts"].to_numpy()
    base["studied_credits"] = info["studied_credits"].to_numpy()
    base["final_result"] = info["final_result"].to_numpy()
    features = base.merge(activity, on=keys, how="left")

    totals = student_vle.groupby(keys)["sum_click"].sum().rename("total_clicks")
    before = student_vle.loc[student_vle["date"] < 0].groupby(keys)["sum_click"].sum().rename("BC_total_clicks")
    after = student_vle.loc[student_vle["date"] >= 0].groupby(keys)["sum_click"].sum().rename("AC_total_clicks")
    features = features.merge(pd.concat([totals, before, after], axis=1).reset_index(), on=keys, how="left")

    relative_days = []
    for shift, label in [(-1, "Before_As_Clicks"), (0, "On_As_Clicks"), (1, "After_As_Clicks")]:
        part = assessments.copy()
        part["date"] = part["date"].astype("int16") + shift
        part["assessment_period"] = label
        relative_days.append(part)
    # Preserve the source pipeline's if/elif priority when assessment windows
    # overlap: before-day first, then on-day, then after-day.
    assessment_days = pd.concat(relative_days, ignore_index=True).drop_duplicates(
        ["code_module", "code_presentation", "date"], keep="first"
    )
    assessment_clicks = (
        student_vle.merge(assessment_days, on=["code_module", "code_presentation", "date"], how="inner")
        .groupby(keys + ["assessment_period"])["sum_click"].sum().unstack(fill_value=0).reset_index()
    )
    features = features.merge(assessment_clicks, on=keys, how="left").fillna(0)

    scenarios = {
        "pass-fail": (
            features[features["final_result"] != "Withdrawn"],
            {"Pass": 0, "Distinction": 0, "Fail": 1}, 22437,
        ),
        "fail-distinction": (
            features[features["final_result"].isin(["Fail", "Distinction"])],
            {"Fail": 0, "Distinction": 1}, 10076,
        ),
        "pass-distinction": (
            features[features["final_result"].isin(["Pass", "Distinction"])],
            {"Pass": 0, "Distinction": 1}, 15385,
        ),
        "pass-withdrawn": (
            features[features["final_result"] != "Fail"],
            {"Pass": 0, "Distinction": 0, "Withdrawn": 1}, 25541,
        ),
    }
    for name, (frame, mapping, expected_rows) in scenarios.items():
        assert len(frame) == expected_rows, (name, len(frame), expected_rows)
        X = frame.drop(columns=keys + ["final_result"])
        y = frame["final_result"].map(mapping).astype("int64").rename("target")
        X.to_csv(PROCESSED_DIR / f"oulad_{name}_x.csv", index=False)
        y.to_csv(PROCESSED_DIR / f"oulad_{name}_y.csv", index=False)
    del student_vle
    return scenarios.keys()


expected_processed = [PROCESSED_DIR / f"oulad_{task}_x.csv" for task in TASKS]
if REBUILD_FEATURES or not all(path.exists() for path in expected_processed):
    engineer_oulad_features()

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)


class DropoutMLP(nn.Module):
    """Two-hidden-layer network used in the paper (30 and 10 units)."""
    def __init__(self, input_dim: int):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 30),
            nn.ReLU(),
            nn.Linear(30, 10),
            nn.ReLU(),
            nn.Linear(10, 2),
        )

    def forward(self, x):
        return self.layers(x)


def inverse_frequency_weights(y, device):
    counts = np.bincount(np.asarray(y, dtype=np.int64), minlength=2)
    if np.any(counts == 0):
        raise ValueError(f"Both classes must be present in the training split; got {counts.tolist()}")
    weights = len(y) / (2.0 * counts)
    return torch.tensor(weights, dtype=torch.float32, device=device)


def make_loader(X, y, batch_size, shuffle, seed):
    dataset = TensorDataset(
        torch.as_tensor(X, dtype=torch.float32),
        torch.as_tensor(y, dtype=torch.long),
    )
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, generator=generator)


def train_central(X, y, epochs, batch_size, lr, seed, device):
    set_seed(seed)
    model = DropoutMLP(X.shape[1]).to(device)
    criterion = nn.CrossEntropyLoss(weight=inverse_frequency_weights(y, device))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = make_loader(X, y, batch_size, True, seed + 1)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
    return model


def partition_clients(X, y, n_clients, seed):
    if n_clients > len(y):
        raise ValueError("The number of clients cannot exceed the number of training rows.")
    rng = np.random.default_rng(seed)
    partitions = np.array_split(rng.permutation(len(y)), n_clients)
    return [(X[idx], y[idx]) for idx in partitions]


def train_federated(X, y, n_clients, rounds, local_epochs, batch_size, lr, mu, seed, device):
    """FedAvg when mu=0, and FedProx when mu>0."""
    set_seed(seed)
    global_model = DropoutMLP(X.shape[1]).to(device)
    clients = partition_clients(X, y, n_clients, seed + 1)
    class_weights = inverse_frequency_weights(y, device)
    client_sizes = np.asarray([len(cy) for _, cy in clients], dtype=np.float64)
    aggregation_weights = client_sizes / client_sizes.sum()

    for round_idx in range(rounds):
        local_states = []
        for client_idx, (client_X, client_y) in enumerate(clients):
            local_model = DropoutMLP(X.shape[1]).to(device)
            local_model.load_state_dict(global_model.state_dict())
            global_reference = [p.detach().clone() for p in global_model.parameters()]
            criterion = nn.CrossEntropyLoss(weight=class_weights)
            optimizer = torch.optim.Adam(local_model.parameters(), lr=lr)
            loader_seed = seed + 10_000 * round_idx + client_idx
            loader = make_loader(client_X, client_y, batch_size, True, loader_seed)

            for _ in range(local_epochs):
                local_model.train()
                for xb, yb in loader:
                    xb, yb = xb.to(device), yb.to(device)
                    optimizer.zero_grad(set_to_none=True)
                    loss = criterion(local_model(xb), yb)
                    if mu > 0:
                        proximal = sum(
                            torch.sum((local - reference) ** 2)
                            for local, reference in zip(local_model.parameters(), global_reference)
                        )
                        loss = loss + (mu / 2.0) * proximal
                    loss.backward()
                    optimizer.step()
            local_states.append({k: v.detach().clone() for k, v in local_model.state_dict().items()})

        aggregated = {}
        for name in global_model.state_dict():
            aggregated[name] = sum(
                float(weight) * state[name]
                for weight, state in zip(aggregation_weights, local_states)
            )
        global_model.load_state_dict(aggregated)
    return global_model


def evaluate(model, X, y, f1_average, device):
    model.eval()
    with torch.no_grad():
        logits = model(torch.as_tensor(X, dtype=torch.float32, device=device))
        probabilities = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        predictions = logits.argmax(dim=1).cpu().numpy()
    return {
        "accuracy": accuracy_score(y, predictions),
        "f1": f1_score(y, predictions, average=f1_average, zero_division=0),
        "auc": roc_auc_score(y, probabilities),
    }


def run_three_methods(X_train, y_train, X_test, y_test, config, f1_average, seed, device):
    central = train_central(
        X_train, y_train, config["central_epochs"], config["batch_size"],
        config["lr"], seed + 100, device,
    )
    fedavg = train_federated(
        X_train, y_train, config["clients"], config["rounds"],
        config["local_epochs"], config["batch_size"], config["lr"],
        0.0, seed + 200, device,
    )
    fedprox = train_federated(
        X_train, y_train, config["clients"], config["rounds"],
        config["local_epochs"], config["batch_size"], config["lr"],
        config["mu"], seed + 200, device,
    )
    models = {"Central": central, "FedAvg": fedavg, "FedProx": fedprox}
    metrics = {name: evaluate(model, X_test, y_test, f1_average, device) for name, model in models.items()}
    return metrics, models

## 2. Run the selected OULAD tasks

In [ ]:
all_rows = []
last_context = None
for task in TASKS:
    X_frame = pd.read_csv(PROCESSED_DIR / f"oulad_{task}_x.csv")
    y = pd.read_csv(PROCESSED_DIR / f"oulad_{task}_y.csv").iloc[:, 0].to_numpy(dtype=np.int64)
    X = X_frame.to_numpy(dtype=np.float32)
    feature_names = X_frame.columns.to_list()

    if CONFIG["fast_max_rows"] and len(y) > CONFIG["fast_max_rows"]:
        selected, _ = train_test_split(
            np.arange(len(y)), train_size=CONFIG["fast_max_rows"], stratify=y, random_state=0
        )
        X, y = X[selected], y[selected]

    for repeat in range(CONFIG["repeats"]):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=repeat
        )
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train).astype(np.float32)
        X_test = scaler.transform(X_test).astype(np.float32)
        metrics, models = run_three_methods(
            X_train, y_train, X_test, y_test, CONFIG, "weighted", repeat, DEVICE
        )
        for method, values in metrics.items():
            all_rows.append({"task": task, "repeat": repeat, "method": method, **values})
        last_context = {
            "model": models["FedProx"], "X_train": X_train, "X_test": X_test,
            "feature_names": feature_names,
        }

run_results = pd.DataFrame(all_rows)
summary = run_results.groupby(["task", "method"])[["accuracy", "f1", "auc"]].agg(["mean", "std"])
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
run_results.to_csv(RESULTS_DIR / "oulad_metrics.csv", index=False)
torch.save(last_context["model"].state_dict(), RESULTS_DIR / "oulad_fedprox_last_seed.pt")
display(summary.round(4))

## 3. Published OULAD results

In [ ]:
published = pd.DataFrame(
    [
        ["pass-fail", "Central", .827, .822], ["pass-fail", "FedAvg", .811, .813], ["pass-fail", "FedProx", .830, .827],
        ["fail-distinction", "Central", .849, .848], ["fail-distinction", "FedAvg", .852, .855], ["fail-distinction", "FedProx", .862, .861],
        ["pass-distinction", "Central", .803, .716], ["pass-distinction", "FedAvg", .716, .730], ["pass-distinction", "FedProx", .804, .756],
        ["pass-withdrawn", "Central", .903, .924], ["pass-withdrawn", "FedAvg", .892, .893], ["pass-withdrawn", "FedProx", .892, .892],
    ],
    columns=["task", "method", "accuracy", "f1"],
)
display(published.set_index(["task", "method"]).style.set_caption("Published weighted-F1 results"))

## 4. Local explanations for the last selected task

In [ ]:
def explain_with_captum(context, sample_index=10, top_k=15):
    """Plot local LIME, Integrated Gradients, and Gradient SHAP attributions."""
    if not RUN_XAI:
        print("XAI skipped. Set AIED_RUN_XAI=1 before launching Jupyter to run this section.")
        return None
    from captum.attr import GradientShap, IntegratedGradients, Lime

    model = context["model"].to(DEVICE).eval()
    X_train = context["X_train"]
    X_test = context["X_test"]
    feature_names = np.asarray(context["feature_names"])
    sample_index = min(sample_index, len(X_test) - 1)
    sample = torch.as_tensor(X_test[[sample_index]], dtype=torch.float32, device=DEVICE)
    background = torch.as_tensor(X_train[: min(64, len(X_train))], dtype=torch.float32, device=DEVICE)

    ig_values = IntegratedGradients(model).attribute(sample, baselines=torch.zeros_like(sample), target=1)
    gs_values = GradientShap(model).attribute(sample, baselines=background, target=1, n_samples=50)
    lime_values = Lime(model).attribute(
        sample, target=1, n_samples=200, perturbations_per_eval=32,
        baselines=torch.zeros_like(sample),
    )
    values = {
        "LIME": lime_values.detach().cpu().numpy().ravel(),
        "Integrated Gradients": ig_values.detach().cpu().numpy().ravel(),
        "Gradient SHAP": gs_values.detach().cpu().numpy().ravel(),
    }

    fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
    for ax, (name, attribution) in zip(axes, values.items()):
        idx = np.argsort(np.abs(attribution))[-top_k:]
        colors = np.where(attribution[idx] >= 0, "#2f6b9a", "#d18f00")
        ax.barh(feature_names[idx], attribution[idx], color=colors)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title(name)
        ax.set_xlabel("Attribution toward dropout/positive class")
    prediction = torch.softmax(model(sample), dim=1)[0, 1].item()
    fig.suptitle(f"Sample {sample_index}: positive-class probability = {prediction:.3f}")
    plt.show()
    return values

In [ ]:
explain_with_captum(last_context, sample_index=10)